# Module 18 - Web Scraping Drills


```
pip install requests beautifulsoup4
```

---

## Parts

- **Part A** — HTML Parsing with BeautifulSoup (exercises 1–4)
- **Part B** — Making HTTP Requests (exercises 5–7)
- **Part C** — Extracting and Cleaning Data (exercises 8–11)
- **Part D** — Pagination and Rate Limiting (exercises 12–13)
- **Part E** — Putting It Together (exercises 14–15)

---

## Setup

Run this cell first. It defines the mock HTML pages and imports used throughout the drills.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time, csv, json, re, io

# ── Mock HTML pages ────────────────────────────────────────────────────────────

MOCK_VULN_PAGE = """
<html>
<body>
  <h1 class="page-title">Vulnerability Report — Q4 2024</h1>
  <div class="vuln-list">
    <div class="vuln-entry critical" id="cve-2024-5001">
      <h2 class="cve-id">CVE-2024-5001</h2>
      <span class="severity critical">CRITICAL</span>
      <span class="cvss-score">9.8</span>
      <p class="affected-product">OpenSSL 3.0.x</p>
      <a href="/vuln/CVE-2024-5001" class="detail-link">View Details</a>
    </div>
    <div class="vuln-entry high" id="cve-2024-5002">
      <h2 class="cve-id">CVE-2024-5002</h2>
      <span class="severity high">HIGH</span>
      <span class="cvss-score">7.5</span>
      <p class="affected-product">Apache HTTP Server 2.4</p>
      <a href="/vuln/CVE-2024-5002" class="detail-link">View Details</a>
    </div>
    <div class="vuln-entry high resolved" id="cve-2024-5003">
      <h2 class="cve-id">CVE-2024-5003</h2>
      <span class="severity high">HIGH</span>
      <span class="cvss-score">7.2</span>
      <p class="affected-product">nginx 1.25.x</p>
      <a href="/vuln/CVE-2024-5003" class="detail-link">View Details</a>
    </div>
    <div class="vuln-entry medium" id="cve-2024-5004">
      <h2 class="cve-id">CVE-2024-5004</h2>
      <span class="severity medium">MEDIUM</span>
      <span class="cvss-score">5.5</span>
      <p class="affected-product">Linux Kernel 6.5</p>
      <a href="/vuln/CVE-2024-5004" class="detail-link">View Details</a>
    </div>
    <div class="vuln-entry low" id="cve-2024-5005">
      <h2 class="cve-id">CVE-2024-5005</h2>
      <span class="severity low">LOW</span>
      <span class="cvss-score">2.1</span>
      <p class="affected-product">curl 8.4.0</p>
      <!-- detail-link is missing on this entry — handle it gracefully -->
    </div>
  </div>
</body>
</html>
"""

MOCK_IP_TABLE = """
<html><body>
<table class="threat-table">
  <thead>
    <tr>
      <th>IP Address</th>
      <th>Threat Category</th>
      <th>Confidence Score</th>
      <th>Country</th>
      <th>Last Reported</th>
    </tr>
  </thead>
  <tbody>
    <tr class="threat-row">
      <td class="ip">45.33.32.156</td>
      <td class="category">SSH Scanner</td>
      <td class="confidence">98</td>
      <td class="country">US</td>
      <td class="last-seen">2024-11-01</td>
    </tr>
    <tr class="threat-row">
      <td class="ip">192.241.187.88</td>
      <td class="category">C2 Server</td>
      <td class="confidence">87</td>
      <td class="country">DE</td>
      <td class="last-seen">2024-10-28</td>
    </tr>
    <tr class="threat-row">
      <td class="ip">104.21.14.101</td>
      <td class="category">Phishing Host</td>
      <td class="confidence">91</td>
      <td class="country">NL</td>
      <td class="last-seen">2024-11-03</td>
    </tr>
    <tr class="threat-row">
      <td class="ip">185.220.101.5</td>
      <td class="category">Tor Exit Node</td>
      <td class="confidence">99</td>
      <td class="country">AT</td>
      <td class="last-seen">2024-11-04</td>
    </tr>
  </tbody>
</table>
</body></html>
"""

MOCK_CT_LOG = """
<html><body>
<div class="ct-results">
  <div class="ct-entry">
    <span class="ct-domain">mail.legitcorp.com</span>
    <span class="ct-issuer">DigiCert Inc</span>
    <span class="ct-logged">2024-11-01 07:00</span>
  </div>
  <div class="ct-entry">
    <span class="ct-domain">secure-login.legitcorp.com.phishing-site.xyz</span>
    <span class="ct-issuer">Let's Encrypt</span>
    <span class="ct-logged">2024-11-01 09:44</span>
  </div>
  <div class="ct-entry">
    <span class="ct-domain">vpn.legitcorp.com</span>
    <span class="ct-issuer">GlobalSign</span>
    <span class="ct-logged">2024-11-01 11:22</span>
  </div>
  <div class="ct-entry">
    <span class="ct-domain">legitcorp.com.malicious-redirect.ru</span>
    <span class="ct-issuer">ZeroSSL</span>
    <span class="ct-logged">2024-11-01 14:05</span>
  </div>
</div>
</body></html>
"""

# Parse the main vulnerability page once — reuse in exercises
soup_vuln = BeautifulSoup(MOCK_VULN_PAGE, "html.parser")
soup_ip   = BeautifulSoup(MOCK_IP_TABLE, "html.parser")
soup_ct   = BeautifulSoup(MOCK_CT_LOG, "html.parser")

print("Setup complete. Mock pages parsed.")
print(f"  soup_vuln : {len(soup_vuln.find_all())} total elements")
print(f"  soup_ip   : {len(soup_ip.find_all())} total elements")
print(f"  soup_ct   : {len(soup_ct.find_all())} total elements")

---

## Part A — HTML Parsing with BeautifulSoup

**Exercise 1**

Using `soup_vuln`:
1. Find the page title using `find()` on the `h1` tag — store its text in `page_title`
2. Find all `<div>` elements with class `"vuln-entry"` — store the list in `all_entries`
3. Print `page_title` and the count of entries found

In [ ]:
# Exercise 1


**Exercise 2**

Using `soup_vuln`:
1. Use `select()` to find all CVE ID elements (class `"cve-id"`)
2. Extract the text from each and store as a list called `cve_ids`
3. Print the list

In [ ]:
# Exercise 2


**Exercise 3**

Using `soup_vuln`, use `select()` with the `:not()` pseudo-class to find all `.vuln-entry` divs
that do **not** have the class `"resolved"`.
Store the count in `unresolved_count` and print it.

In [ ]:
# Exercise 3


**Exercise 4**

Using `soup_vuln`, extract all `href` values from links with class `"detail-link"`.

One entry does NOT have a detail-link. Use `tag.get("href")` (not `tag["href"]`)
so your code does not crash on the missing entry.

Store all collected hrefs (skip `None` values) in a list called `detail_hrefs`. Print it.

In [ ]:
# Exercise 4


---

## Part B — Making HTTP Requests

**Exercise 5**

Fetch the page at `https://httpbin.org/get` using `requests.get()`.

Store in variables:
- `status_code` — the HTTP status code (integer)
- `content_type` — the `Content-Type` response header value

Print both. Use a 10-second timeout.

In [ ]:
# Exercise 5


**Exercise 6**

Write a function `fetch_page(url)` that:
1. Uses a `requests.Session()` with a descriptive User-Agent
2. Sets a 10-second timeout
3. Calls `raise_for_status()`
4. Returns the `response.text` on success
5. Returns `None` on any `requests.exceptions.RequestException`

Test it with `https://httpbin.org/html` (should succeed) and `https://httpbin.org/status/403` (should fail and return `None`).

In [ ]:
# Exercise 6


**Exercise 7**

Using `fetch_page()` from Exercise 6:
1. Fetch `https://httpbin.org/html`
2. Parse the result with BeautifulSoup
3. Find the first `<h1>` tag — store its text in `fetched_heading`
4. Print `fetched_heading`

In [ ]:
# Exercise 7


---

## Part C — Extracting and Cleaning Data

**Exercise 8**

Using `soup_vuln`, loop through all `.vuln-entry` divs and build a list of dicts called `vuln_records`.
Each dict should have these keys: `cve_id`, `severity`, `cvss_score` (as `float`), `product`.

Print the list.

In [ ]:
# Exercise 8


**Exercise 9**

Using `soup_ip`, extract all rows from the threat table into a list of dicts called `ip_records`.
Each dict: `ip`, `category`, `confidence` (as `int`), `country`, `last_seen`.

Print each record on one line in this format:
```
45.33.32.156  SSH Scanner     98  US  2024-11-01
```

In [ ]:
# Exercise 9


**Exercise 10**

Using `vuln_records` from Exercise 8:
1. Filter entries where `cvss_score >= 7.0` into a list called `high_risk`
2. Sort `high_risk` by `cvss_score` descending (highest first)
3. Print the CVE ID and score for each

Hint: use the `sorted()` built-in with `key=` and `reverse=True`.

In [ ]:
# Exercise 10


**Exercise 11**

Using `soup_ct`, extract all CT log entries into a list of dicts called `ct_records`
with keys: `domain`, `issuer`, `logged_at`.

Then apply the following rules and store results:
- Entries where the domain contains more than one `.` after the main domain portion AND
  appears to be a subdomain impersonation — store suspicious domain strings in `suspicious_domains`
- A simple heuristic: flag any domain where the original brand name appears in the **middle** of the domain,
  not at the start. For example: `legitcorp.com.phishing-site.xyz` has `legitcorp.com` in the middle.
  Check with `"legitcorp.com" in domain and not domain.startswith("legitcorp.com")`

Print the suspicious domains.

In [ ]:
# Exercise 11


---

## Part D — Pagination and Rate Limiting

**Exercise 12**

The function `mock_feed_page(page)` below returns a mock threat feed page.
Pages 1–3 have data; page 4 is empty.

Write a paginated scraper that:
1. Loops through pages 1 onward
2. Stops when a page has no `.feed-entry` elements
3. Collects all IP strings into `paginated_ips`
4. Adds a `time.sleep(0.3)` delay between pages
5. Prints the total collected at the end

In [ ]:
def mock_feed_page(page):
    data = {
        1: ["10.0.0.1", "10.0.0.2"],
        2: ["172.16.0.5", "172.16.0.6", "172.16.0.7"],
        3: ["192.168.1.100"],
    }
    ips = data.get(page, [])
    rows = "".join(f'<div class="feed-entry">{ip}</div>' for ip in ips)
    return f"<html><body><div class='feed'>{rows}</div></body></html>"

# Your scraping loop here


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
paginated_ips = []
max_pages = 10

for page in range(1, max_pages + 1):
    html = mock_feed_page(page)
    soup_feed = BeautifulSoup(html, "html.parser")

    entries = soup_feed.select("div.feed-entry")
    if not entries:
        print(f"Page {page}: empty — stopping")
        break

    for entry in entries:
        paginated_ips.append(entry.get_text(strip=True))

    print(f"Page {page}: {len(entries)} IPs")
    time.sleep(0.3)

print(f"Total IPs: {len(paginated_ips)} — {paginated_ips}")
```

</details>

**Exercise 13**

Write a function `polite_fetch(urls, session, delay=1.5)` that:
1. Accepts a list of URLs
2. Fetches each one with a `timeout=10`
3. Sleeps `delay` seconds between requests (not after the last one)
4. Returns a list of `(url, status_code)` tuples

Test it with these three URLs:
```python
test_urls = [
    "https://httpbin.org/get",
    "https://httpbin.org/status/200",
    "https://httpbin.org/status/404",
]
```
Print each result tuple.

In [ ]:
# Exercise 13


---

## Part E — Putting It Together

**Exercise 14 — Full Pipeline**

Build a complete scraping pipeline using `MOCK_VULN_PAGE`:

1. Parse the HTML with BeautifulSoup
2. Extract all vulnerability entries into a list of dicts with keys:
   `cve_id`, `severity`, `cvss_score` (float), `product`, `detail_url` (use `.get()`, may be `None`)
3. Filter to only entries with `cvss_score >= 7.0`
4. Save the filtered list to `critical_findings.json` with `indent=2`
5. Read it back and print the count and each CVE ID

Store the final filtered list in `critical_findings`.

In [ ]:
# Exercise 14


**Exercise 15 — Defensive Scraping**

A production scraper must handle missing fields without crashing.
The `MOCK_BROKEN_PAGE` below has intentionally malformed records.

Parse it and extract records into `extracted_records` — a list of dicts with keys
`cve_id`, `cvss_score`. For missing fields, use sensible defaults:
- Missing CVE ID: `"UNKNOWN"`
- Missing score: `None`

After extraction, print each record and a count of records where score is `None`.

In [ ]:
MOCK_BROKEN_PAGE = """
<html><body>
<div class="vuln-entry">
  <h2 class="cve-id">CVE-2024-9001</h2>
  <span class="cvss-score">8.5</span>
</div>
<div class="vuln-entry">
  <!-- cve-id missing -->
  <span class="cvss-score">6.2</span>
</div>
<div class="vuln-entry">
  <h2 class="cve-id">CVE-2024-9003</h2>
  <!-- cvss-score missing -->
</div>
<div class="vuln-entry">
  <!-- both fields missing -->
</div>
</body></html>
"""

# Your code here


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
soup_broken = BeautifulSoup(MOCK_BROKEN_PAGE, "html.parser")

extracted_records = []
for entry in soup_broken.find_all("div", class_="vuln-entry"):
    cve_tag   = entry.find("h2", class_="cve-id")
    score_tag = entry.find("span", class_="cvss-score")

    cve_id = cve_tag.get_text(strip=True) if cve_tag else "UNKNOWN"
    score  = float(score_tag.get_text(strip=True)) if score_tag else None

    extracted_records.append({"cve_id": cve_id, "cvss_score": score})

for rec in extracted_records:
    print(rec)

missing_scores = sum(1 for r in extracted_records if r["cvss_score"] is None)
print(f"\nRecords with missing score: {missing_scores}/{len(extracted_records)}")
```

</details>

---

## Tests

> **Note:** The test cell uses Python classes (`class TestScraping`). Classes are covered in the OOP module.
> You do not need to understand this syntax — just run the cell and read the output.

Run all exercises first, then run this cell. Each passing test prints `ok`.

In [ ]:
import unittest

class TestScraping(unittest.TestCase):

    # ── Part A ──────────────────────────────────────────────────────────────

    def test_01_page_title(self):
        self.assertIsInstance(page_title, str,
                              "page_title must be a str — use tag.text or get_text()")
        self.assertIn("Vulnerability", page_title,
                      "page_title should contain the word 'Vulnerability'")

    def test_02_all_entries_count(self):
        self.assertEqual(len(all_entries), 5,
                         "all_entries should have 5 vuln-entry divs from the mock page")

    def test_03_cve_ids_list(self):
        self.assertIsInstance(cve_ids, list,
                              "cve_ids must be a list")
        self.assertEqual(len(cve_ids), 5,
                         "cve_ids should contain all 5 CVE IDs from the page")
        self.assertIn("CVE-2024-5001", cve_ids,
                      "cve_ids should contain CVE-2024-5001")

    def test_04_unresolved_count(self):
        self.assertEqual(unresolved_count, 4,
                         "unresolved_count should be 4 — one entry has class 'resolved'")

    def test_05_detail_hrefs(self):
        self.assertIsInstance(detail_hrefs, list,
                              "detail_hrefs must be a list")
        self.assertEqual(len(detail_hrefs), 4,
                         "detail_hrefs should have 4 entries — the 5th entry has no link")
        self.assertIn("/vuln/CVE-2024-5001", detail_hrefs,
                      "detail_hrefs should include /vuln/CVE-2024-5001")

    # ── Part B ──────────────────────────────────────────────────────────────

    def test_06_status_code(self):
        self.assertEqual(status_code, 200,
                         "status_code should be 200 for httpbin.org/get")
        self.assertIsInstance(status_code, int,
                              "status_code must be int — response.status_code is already an int")

    def test_07_fetched_heading(self):
        self.assertIsInstance(fetched_heading, str,
                              "fetched_heading must be a str from the h1 tag")
        self.assertGreater(len(fetched_heading), 0,
                           "fetched_heading should not be empty")

    # ── Part C ──────────────────────────────────────────────────────────────

    def test_08_vuln_records(self):
        self.assertIsInstance(vuln_records, list,
                              "vuln_records must be a list of dicts")
        self.assertEqual(len(vuln_records), 5,
                         "vuln_records should have 5 entries")
        first = vuln_records[0]
        self.assertIn("cve_id", first, "each record must have a 'cve_id' key")
        self.assertIn("cvss_score", first, "each record must have a 'cvss_score' key")
        self.assertIsInstance(first["cvss_score"], float,
                              "cvss_score must be float — use float(tag.get_text())")

    def test_09_ip_records(self):
        self.assertIsInstance(ip_records, list,
                              "ip_records must be a list")
        self.assertEqual(len(ip_records), 4,
                         "ip_records should have 4 entries from the mock table")
        self.assertIsInstance(ip_records[0]["confidence"], int,
                              "confidence must be int — convert with int()")

    def test_10_high_risk(self):
        self.assertIsInstance(high_risk, list,
                              "high_risk must be a list")
        self.assertEqual(len(high_risk), 3,
                         "high_risk should have 3 entries with cvss_score >= 7.0")
        scores = [r["cvss_score"] for r in high_risk]
        self.assertEqual(scores, sorted(scores, reverse=True),
                         "high_risk must be sorted by cvss_score descending")

    def test_11_ct_records(self):
        self.assertIsInstance(ct_records, list,
                              "ct_records must be a list")
        self.assertEqual(len(ct_records), 4,
                         "ct_records should have 4 entries from the CT log")

    def test_12_suspicious_domains(self):
        self.assertIsInstance(suspicious_domains, list,
                              "suspicious_domains must be a list")
        self.assertEqual(len(suspicious_domains), 2,
                         "There are 2 suspicious domains in the CT log mock")
        for d in suspicious_domains:
            self.assertIn("legitcorp.com", d,
                          "Each suspicious domain should contain 'legitcorp.com'")

    # ── Part D ──────────────────────────────────────────────────────────────

    def test_13_paginated_ips(self):
        self.assertIsInstance(paginated_ips, list,
                              "paginated_ips must be a list")
        self.assertEqual(len(paginated_ips), 6,
                         "paginated_ips should have 6 IPs across 3 pages (2+3+1)")

    # ── Part E ──────────────────────────────────────────────────────────────

    def test_14_critical_findings(self):
        self.assertIsInstance(critical_findings, list,
                              "critical_findings must be a list")
        self.assertEqual(len(critical_findings), 3,
                         "critical_findings should have 3 entries (CVSS >= 7.0)")
        for entry in critical_findings:
            self.assertGreaterEqual(entry["cvss_score"], 7.0,
                                    "All critical_findings must have cvss_score >= 7.0")

    def test_15_extracted_records(self):
        self.assertIsInstance(extracted_records, list,
                              "extracted_records must be a list")
        self.assertEqual(len(extracted_records), 4,
                         "extracted_records should have 4 entries from MOCK_BROKEN_PAGE")
        # Check default values used for missing fields
        cve_ids_extracted = [r["cve_id"] for r in extracted_records]
        self.assertIn("UNKNOWN", cve_ids_extracted,
                      "Records with missing cve-id should use 'UNKNOWN'")
        scores = [r["cvss_score"] for r in extracted_records]
        self.assertIn(None, scores,
                      "Records with missing score should use None")


unittest.main(argv=[""], verbosity=2, exit=False)

---

## All tests passed?

You have completed the full Python curriculum.

**You can now:**
- Automatically collect vulnerability data, IP intelligence, and domain intelligence from the web
- Handle real-world scraping challenges: pagination, missing fields, server errors, rate limiting
- Write scraping scripts that are professional, robust, and legally aware

**Next:** Apply these skills in the module capstone project or your track specialisation.